# Phase 41 — BERT-large + BERT-multilingual + Focal Loss

## Μοντέλο 1: bert-large-uncased
- 336M parameters (vs 110M του base)
- 24 layers, 16 attention heads
- Αναμένουμε καλύτερο από BERT-base (0.80399)

## Μοντέλο 2: bert-base-multilingual-uncased
- 110M parameters, εκπαιδευμένο σε 102 γλώσσες
- Χειρίζεται τα non-English δείγματα του dataset

**Best so far:** 0.81882

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_full_product = le_product.transform(train_full['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Train+Valid: {len(train_full)} | Test: {len(texts_test)}')
print(f'Hazard: {n_hazard} | Product: {n_product}')

# Ανάλυση γλωσσών στο dataset
print('\n=== ΑΝΑΛΥΣΗ ΧΩΡΩΝ/ΓΛΩΣΣΩΝ ===')
print(train_full['country'].value_counts().head(10).to_string())

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, reduction='none')
        pt    = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

criterion = FocalLoss(gamma=2.0)


class FoodHazardDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts; self.labels = labels
        self.tokenizer = tokenizer; self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_length,
                              padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label':          torch.tensor(self.labels[idx], dtype=torch.long)}


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        labels = batch.pop('label').to(device)
        batch  = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = criterion(model(**batch).logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def get_probabilities(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            probs = torch.softmax(model(**batch).logits, dim=1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)


def official_st1_score(y_true_h, y_pred_h, y_true_p, y_pred_p, verbose=True):
    f1_h = f1_score(y_true_h, y_pred_h, average='macro', zero_division=0)
    mask = (np.array(y_true_h) == np.array(y_pred_h))
    f1_p = f1_score(
        np.array(y_true_p)[mask], np.array(y_pred_p)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_h + f1_p) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_h:.4f}')
        print(f'  Σωστά hazard:                       {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_p:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score


def train_and_save(model_name, batch_size, lr, n_epochs, suffix):
    """Εκπαιδεύει ένα μοντέλο και αποθηκεύει probs + submission."""
    print(f'\n{"="*60}')
    print(f'ΜΟΝΤΕΛΟ: {model_name}')
    print(f'batch_size={batch_size} | LR={lr} | epochs={n_epochs}')
    print(f'{"="*60}\n')

    tokenizer    = AutoTokenizer.from_pretrained(model_name)
    dummy_labels = np.zeros(len(texts_test), dtype=int)

    full_loader_h = DataLoader(FoodHazardDataset(texts_full, y_full_hazard,  tokenizer, 128), batch_size=batch_size, shuffle=True)
    full_loader_p = DataLoader(FoodHazardDataset(texts_full, y_full_product, tokenizer, 128), batch_size=batch_size, shuffle=True)
    test_loader_h = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, 128), batch_size=batch_size, shuffle=False)
    test_loader_p = DataLoader(FoodHazardDataset(texts_test, dummy_labels,   tokenizer, 128), batch_size=batch_size, shuffle=False)

    # Hazard
    print(f'--- Hazard Classifier ---')
    model_h  = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_hazard).to(device)
    opt_h    = AdamW(model_h.parameters(), lr=lr, weight_decay=0.01)
    steps_h  = len(full_loader_h) * n_epochs
    sch_h    = get_linear_schedule_with_warmup(opt_h, int(steps_h*0.1), steps_h)
    for epoch in range(n_epochs):
        loss = train_epoch(model_h, full_loader_h, opt_h, sch_h)
        print(f'Epoch {epoch+1}/{n_epochs} | Loss: {loss:.4f}')

    # Product
    print(f'\n--- Product Classifier ---')
    model_p  = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_product).to(device)
    opt_p    = AdamW(model_p.parameters(), lr=lr, weight_decay=0.01)
    steps_p  = len(full_loader_p) * n_epochs
    sch_p    = get_linear_schedule_with_warmup(opt_p, int(steps_p*0.1), steps_p)
    for epoch in range(n_epochs):
        loss = train_epoch(model_p, full_loader_p, opt_p, sch_p)
        print(f'Epoch {epoch+1}/{n_epochs} | Loss: {loss:.4f}')

    # Probs & Submission
    hazard_probs  = get_probabilities(model_h, test_loader_h)
    product_probs = get_probabilities(model_p, test_loader_p)

    np.save(f'{suffix}_test_hazard_probs.npy',  hazard_probs)
    np.save(f'{suffix}_test_product_probs.npy', product_probs)

    test_hazard  = le_hazard.inverse_transform(hazard_probs.argmax(axis=1))
    test_product = le_product.inverse_transform(product_probs.argmax(axis=1))

    pd.DataFrame({
        'id': test['id'],
        'hazard-category': test_hazard,
        'product-category': test_product
    }).to_csv(f'submission_{suffix}.csv', index=False)

    print(f'\nsubmission_{suffix}.csv')
    print(f'{suffix}_test_hazard_probs.npy')
    print(f'{suffix}_test_product_probs.npy')

    # Καθαρισμός VRAM
    del model_h, model_p
    torch.cuda.empty_cache()

    return hazard_probs, product_probs

## Μοντέλο 1: BERT-large-uncased + Focal Loss

- 336M parameters
- LR=1e-5 (μικρότερο για large μοντέλο)
- BATCH_SIZE=16 (λόγω μεγέθους)

In [ ]:
bert_large_hazard_probs, bert_large_product_probs = train_and_save(
    model_name = 'bert-large-uncased',
    batch_size = 16,
    lr         = 1e-5,
    n_epochs   = 20,
    suffix     = 'bert_large'
)

## Μοντέλο 2: BERT-base-multilingual-uncased + Focal Loss

- 110M parameters, 102 γλώσσες
- Χειρίζεται non-English δείγματα
- LR=2e-5 (ίδιο με BERT-base)

In [ ]:
bert_multi_hazard_probs, bert_multi_product_probs = train_and_save(
    model_name = 'bert-base-multilingual-uncased',
    batch_size = 32,
    lr         = 2e-5,
    n_epochs   = 20,
    suffix     = 'bert_multilingual'
)

## Ensemble: BERT-large + BERT-base (best) + Multi-task

In [ ]:
# Φόρτωση υπαρχόντων best probs
bertbase_hazard_probs  = np.load('bertbase_focal_test_hazard_probs.npy')
bertbase_product_probs = np.load('bertbase_focal_test_product_probs.npy')
multi_hazard_probs     = np.load('multitask_test_hazard_probs.npy')
multi_product_probs    = np.load('multitask_test_product_probs.npy')

# Δοκιμή ensemble combinations
print('=== ENSEMBLE BERT-large + BERT-base + Multi-task ===\n')

combinations = [
    # (w_large, w_base, w_multi)
    (1.0, 0.0, 0.0),   # Large μόνο
    (0.5, 0.5, 0.0),   # Large + Base
    (0.4, 0.3, 0.3),   # Ισορροπημένο
    (0.5, 0.25, 0.25), # Large dominant
    (0.33, 0.33, 0.34),# Equal
    (0.6, 0.2, 0.2),   # Large heavy
]

for w_l, w_b, w_m in combinations:
    h_avg = w_l * bert_large_hazard_probs  + w_b * bertbase_hazard_probs  + w_m * multi_hazard_probs
    p_avg = w_l * bert_large_product_probs + w_b * bertbase_product_probs + w_m * multi_product_probs

    test_h = le_hazard.inverse_transform(h_avg.argmax(axis=1))
    test_p = le_product.inverse_transform(p_avg.argmax(axis=1))

    filename = f'submission_ensemble_L{int(w_l*10)}_B{int(w_b*10)}_M{int(w_m*10)}.csv'
    pd.DataFrame({
        'id': test['id'],
        'hazard-category': test_h,
        'product-category': test_p
    }).to_csv(filename, index=False)
    print(f'Large={w_l}, Base={w_b}, Multi={w_m} → {filename}')

print('\n=== ΣΥΓΚΡΙΣΗ ===')
print('BERT-base Focal:         0.80399')
print('Best ensemble so far:    0.81882')